# PyMessage Walkthrough

A hands-on tour of every function in the package, running entirely against the **built-in example backup** — no real iPhone needed.

Each section is independent. Run them in order for the smoothest experience, or jump straight to whatever interests you.

In [20]:
from pymessage import (
    EXAMPLE_BACKUP,
    Backup,
    find_backups,
    get_activity_summary,
    get_attachments,
    get_contact_summary,
    get_messages,
)

---
## 1. Discovering Backups — `find_backups()`

`find_backups()` scans two places on your Mac:
- `~/Library/Application Support/MobileSync/Backup/` — iPhone backups
- `~/Library/Messages/chat.db` — macOS Messages app

It returns a plain `list[Backup]` sorted by most recent first.

In [30]:
backups = find_backups()

if backups:
    for b in backups:
        print(b)
else:
    print("No backups found on this machine.")

[macOS] MacBook Messages


The `Backup` dataclass wraps everything needed to open a database:

| Field | Description |
|-------|-------------|
| `type` | `"iphone"` or `"macos"` |
| `path` | Backup directory (iphone) or path to `chat.db` (macos) |
| `device_name` | Human-readable label |
| `last_backup` | Datetime of most recent backup |
| `ios_version` | e.g. `"17.2"` |
| `phone_number` | Device phone number |

`EXAMPLE_BACKUP` is a fake iPhone backup built into the package for testing and demos:

In [32]:
print(EXAMPLE_BACKUP)
print()
print(f"type         = {EXAMPLE_BACKUP.type!r}")
print(f"device_name  = {EXAMPLE_BACKUP.device_name!r}")
print(f"ios_version  = {EXAMPLE_BACKUP.ios_version!r}")
print(f"phone_number = {EXAMPLE_BACKUP.phone_number!r}")
print(f"last_backup  = {EXAMPLE_BACKUP.last_backup}")
print(f"path         = {EXAMPLE_BACKUP.path}")

[iPhone] Tucker's iPhone (iOS 17.2) — Last backup: 2024-03-01

type         = 'iphone'
device_name  = "Tucker's iPhone"
ios_version  = '17.2'
phone_number = '+18015550101'
last_backup  = 2024-03-01 00:00:00
path         = /Users/tuckertrost/Documents/School/Package Dev/PyMessage/src/pymessage/data/example_backup


---
## 2. Getting Messages — `get_messages()`

`get_messages(backup)` queries the iMessage database and returns a DataFrame where every row is one message (or tapback reaction).

In [21]:
df = get_messages(EXAMPLE_BACKUP)

df.head()

,timestamp,read_at,sender,contact_name,message_text,is_from_me,chat_id,is_group_chat,attachment_path,reaction_type,reaction_action
0,2026-01-13 09:05:00+00:00,2026-01-13 09:06:00+00:00,+18015550001,Brother Hathaway,"Hey everyone, welcome to the CS Package Dev gr...",False,chat192837465,True,NaN,NaN,NaN
1,2026-01-13 09:12:00+00:00,NaT,NaN,Me,Thanks for setting this up!,True,chat192837465,True,NaN,NaN,NaN
2,2026-01-13 09:15:00+00:00,2026-01-13 09:16:00+00:00,+18015550002,Caleb,Finally a group chat without memes (hopefully),False,chat192837465,True,NaN,NaN,NaN
3,2026-01-13 09:17:00+00:00,2026-01-13 09:18:00+00:00,+18015550003,Mitch,No promises 😂,False,chat192837465,True,NaN,NaN,NaN
4,2026-01-16 08:01:00+00:00,2026-01-16 08:02:00+00:00,+18015550001,Brother Hathaway,Tucker. Status update on the module?,False,+18015550001,False,NaN,NaN,NaN


### `contact_name` — display names from the handle table

- Sent messages (`is_from_me=True`) → `"Me"`
- Received messages → resolved from `handle.uncanonicalized_id` (the display name iMessage stores)

In [22]:
sent     = df[df["is_from_me"]]
received = df[~df["is_from_me"]]

print(f"Sent messages:     {len(sent):>3}  (contact_name = 'Me')")
print(f"Received messages: {len(received):>3}  (contact_name = sender's display name)")
print()
print("All contacts:", sorted(df["contact_name"].unique().tolist()))

Sent messages:      35  (contact_name = 'Me')
Received messages:  70  (contact_name = sender's display name)

All contacts: ['Alyssa', 'Brother Hathaway', 'Caleb', 'Dallin', 'Emma', 'John', 'Me', 'Mitch', 'Phillip', 'Ryan']


### Tapback reactions

Reactions are stored as separate messages in iMessage. `reaction_type` and `reaction_action` are parsed from `associated_message_type` (codes 2000–2007 = add, 3000–3007 = remove).

In [36]:
reactions = df[df["reaction_type"].notna()][
    ["contact_name", "reaction_type", "reaction_action", "message_text"]
]
reactions

,contact_name,reaction_type,reaction_action,message_text
12,Mitch,laughed,add,"Laughed at ""honestly not that surprising"""
26,Caleb,loved,add,"Loved ""I mean I did write the numpy docs so"""
30,Mitch,loved,add,👀
47,Mitch,loved,add,"Loved ""the $65/hr helps lol"""
48,John,loved,add,"Loved ""the $65/hr helps lol"""
56,Dallin,liked,add,"Liked ""workshop is TOMORROW"""
62,Caleb,liked,add,"Liked ""congrats"""
68,Me,loved,add,"Loved ""just reshape with np.newaxis"""
88,Alyssa,loved,add,👀
104,Mitch,liked,add,"Liked ""Good work everyone"""


---
## 3. Filtering Messages

All filters are optional and composable.

### Filter by phone number

In [38]:
caleb_df = get_messages(EXAMPLE_BACKUP, phone_numbers="+18015550002")

print(f"Messages with Caleb: {len(caleb_df)}")
caleb_df[["timestamp", "contact_name", "message_text"]].head(6)

Messages with Caleb: 15


,timestamp,contact_name,message_text
0,2026-01-21 18:30:00+00:00,Caleb,Dude graduation is like 3 weeks away. feels un...
1,2026-01-21 18:32:00+00:00,Me,right?? feels like we just started
2,2026-01-21 18:35:00+00:00,Caleb,I'm lowkey nervous about the job
3,2026-01-21 18:38:00+00:00,Me,You'll crush it. Arkansas will be lucky to hav...
4,2026-01-21 18:40:00+00:00,Caleb,Tucker. Arkansas is actually really nice.
5,2026-01-21 18:41:00+00:00,Me,I'm sure the Walmart has great lighting


### Filter by multiple contacts

In [39]:
multi_df = get_messages(
    EXAMPLE_BACKUP,
    phone_numbers=["+18015550002", "+18015550003"],  # Caleb + Mitch
)
print(f"Messages with Caleb or Mitch: {len(multi_df)}")
multi_df["contact_name"].value_counts()

Messages with Caleb or Mitch: 26


contact_name
Me       12
Caleb     7
Mitch     7
Name: count, dtype: int64

### Filter by date range

In [41]:
# Build a range from the second half of the data
all_ts   = df["timestamp"]
midpoint = all_ts.min() + (all_ts.max() - all_ts.min()) / 2
start    = midpoint.strftime("%Y-%m-%d")
end      = all_ts.max().strftime("%Y-%m-%d")

recent_df = get_messages(EXAMPLE_BACKUP, date_range=(start, end))

print(f"Full data:  {all_ts.min().date()} → {all_ts.max().date()}  ({len(df)} messages)")
print(f"Filtered:   {start} → {end}  ({len(recent_df)} messages)")
recent_df[["timestamp", "contact_name", "message_text"]].head(5)

Full data:  2026-01-13 → 2026-03-10  (105 messages)
Filtered:   2026-02-10 → 2026-03-10  (37 messages)


,timestamp,contact_name,message_text
0,2026-02-10 10:30:00+00:00,Me,"ran into a numpy broadcasting issue, anyone kn..."
1,2026-02-10 10:31:00+00:00,Mitch,John to the rescue again I assume
2,2026-02-10 10:33:00+00:00,John,yeah just reshape with np.newaxis
3,2026-02-10 10:34:00+00:00,Me,"Loved ""just reshape with np.newaxis"""
4,2026-02-10 10:35:00+00:00,Caleb,See? Royalty.


### Combine filters — phone number + date range

In [42]:
filtered = get_messages(
    EXAMPLE_BACKUP,
    phone_numbers="+18015550002",
    date_range=(start, end),
)
print(f"Caleb messages in date range: {len(filtered)}")
filtered[["timestamp", "contact_name", "message_text"]]

Caleb messages in date range: 3


,timestamp,contact_name,message_text
0,2026-02-25 20:00:00+00:00,Me,Have you started packing yet?
1,2026-02-25 20:03:00+00:00,Caleb,lol no. I'm in denial
2,2026-02-25 20:05:00+00:00,Me,same energy tbh


---
## 4. Getting Attachments — `get_attachments()`

Returns metadata for every file shared in messages. For iPhone backups, `file_path` resolves the SHA-1 hashed path inside the backup directory.

In [17]:
attachments = get_attachments(EXAMPLE_BACKUP)

print(f"Attachments: {len(attachments)}")
print(f"Columns: {list(attachments.columns)}")
attachments

Attachments: 1
Columns: ['attachment_id', 'message_id', 'filename', 'mime_type', 'file_size', 'file_path', 'timestamp', 'sender']


,attachment_id,message_id,filename,mime_type,file_size,file_path,timestamp,sender
0,1,49,Library/SMS/Attachments/a1/01/IMG_4821.jpg,image/jpeg,2400000,None,2026-03-07 11:20:00+00:00,None


In [47]:
# Filter attachments to a specific contact
att_filtered = get_attachments(EXAMPLE_BACKUP, phone_numbers="+18015550002")
print(f"Attachments from Caleb: {len(att_filtered)}")

Attachments from Caleb: 0


---
## 5. Activity Summary — `get_activity_summary()`

Computes overall statistics across all messages. Returns `(summary_df, top_contacts_df)`.

Optional: `start`, `end`, `last_n_days`, `reference_date`, `top_n`

In [16]:
summary, top_contacts = get_activity_summary(df)

s = summary.iloc[0]
s

total_messages                                                                 105
total_sent                                                                      35
total_received                                                                  70
avg_messages_per_day                                                      1.842105
unique_contacts                                                                  9
most_active_day_of_week                                                     Monday
most_active_hour                                                                16
late_night_contacts                                                             []
pct_messages_with_attachments                                             0.009524
avg_message_length                                                       36.380952
avg_response_time_seconds                                              1030.344828
conversations_initiated                                                          5
conv

In [49]:
print("=== Top Contacts ===")
top_contacts

=== Top Contacts ===


,contact,total,sent,received
0,Caleb,15,8,7
1,Brother Hathaway,11,4,7
2,Mitch,11,4,7
3,Emma,9,5,4
4,Ryan,6,3,3


### Filter to last N days

In [50]:
import pandas as pd

# Use the data's own max timestamp as the reference so the example always works
ref = df["timestamp"].max()

summary_30, top_30 = get_activity_summary(df, last_n_days=30, reference_date=ref)
print(f"Messages in last 30 days: {summary_30.iloc[0]['total_messages']}")
top_30

Messages in last 30 days: 40


,contact,total,sent,received
0,Emma,9,5,4
1,Ryan,6,3,3
2,Mitch,4,2,2
3,Brother Hathaway,3,1,2
4,Caleb,3,2,1


---
## 6. Per-Contact Stats — `get_contact_summary()`

Returns a single-row DataFrame with detailed stats for one conversation. Accepts the phone number in any format.

In [14]:
cs = get_contact_summary(df, "+18015550002").iloc[0]  # Caleb
cs

total_messages                              15
total_sent                                   8
total_received                               7
send_receive_ratio                    1.142857
avg_messages_per_active_day                5.0
total_active_days                            3
avg_read_time_seconds                     60.0
avg_response_time_you_seconds            120.0
avg_response_time_contact_seconds        132.0
conversations_initiated_you                  1
conversations_initiated_contact              2
longest_gap_days                     21.407639
messages_with_attachments                    0
avg_message_length_you                  35.625
avg_message_length_contact           33.714286
short_message_count_you                      1
short_message_count_contact                  1
most_active_hour                            18
most_active_day_of_week              Wednesday
Name: 0, dtype: object

In [ ]:
# Mitch 
ms = get_contact_summary(df, "+18015550003").iloc[0]
ms

total_messages                                11
total_sent                                     4
total_received                                 7
send_receive_ratio                      0.571429
avg_messages_per_active_day                  5.5
total_active_days                              2
avg_read_time_seconds                3522.857143
avg_response_time_you_seconds             6030.0
avg_response_time_contact_seconds          180.0
conversations_initiated_you                    0
conversations_initiated_contact                2
longest_gap_days                        14.21875
messages_with_attachments                      0
avg_message_length_you                      45.5
avg_message_length_contact                  50.0
short_message_count_you                        1
short_message_count_contact                    1
most_active_hour                               8
most_active_day_of_week                 Thursday
Name: 0, dtype: object

---
## Using Your Own Data

Everything above works identically against a real iPhone backup or macOS Messages database. Just swap `EXAMPLE_BACKUP` for your own:

In [23]:
# Uncomment and run to use your real data:

from pymessage import find_backups, get_messages
backups = find_backups()
print(backups)          # pick the one you want

df = get_messages(backups[0])
df.head()

[[macOS] MacBook Messages]


,timestamp,read_at,sender,contact_name,message_text,is_from_me,chat_id,is_group_chat,attachment_path,reaction_type,reaction_action
0,2018-10-19 20:59:10.912475008+00:00,NaT,NaN,Me,￼,True,+15035229983,False,NaN,NaN,NaN
1,2018-10-19 21:00:10.683527488+00:00,2018-10-19 21:00:11+00:00,+15035229983,Mommy,2+2=4well done guys,False,+15035229983,False,NaN,NaN,NaN
2,2018-10-19 21:00:35.048254016+00:00,2018-10-19 21:00:38+00:00,+19099575449,Kyle Trost,￼￼￼￼￼￼￼,False,+19099575449,False,~/Library/Messages/Attachments/ed/13/at_5_DB0B...,NaN,NaN
3,2018-10-19 21:00:35.048254016+00:00,2018-10-19 21:00:38+00:00,+19099575449,Kyle Trost,￼￼￼￼￼￼￼,False,+19099575449,False,~/Library/Messages/Attachments/ec/12/at_4_DB0B...,NaN,NaN
4,2018-10-19 21:00:35.048254016+00:00,2018-10-19 21:00:38+00:00,+19099575449,Kyle Trost,￼￼￼￼￼￼￼,False,+19099575449,False,~/Library/Messages/Attachments/e8/08/at_0_DB0B...,NaN,NaN


---
## Quick Reference

| Function | Returns | Key optional params |
|----------|---------|---------------------|
| `find_backups()` | `list[Backup]` | — |
| `get_messages(backup)` | `DataFrame` | `phone_numbers`, `date_range`, `output_csv` |
| `get_attachments(backup)` | `DataFrame` | `phone_numbers` |
| `get_activity_summary(df)` | `(summary_df, top_df)` | `start`, `end`, `last_n_days`, `top_n` |
| `get_contact_summary(df, contact)` | `DataFrame` | `start`, `end`, `last_n_days` |